# SongNet: Colab runner
**Runtime > Change runtime type > T4 GPU** before starting.

## Step 2: Get and verify the data

### 1. Mount Google Drive (for saving splits, features, checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/songnet'
os.makedirs(DRIVE_DIR, exist_ok=True)

### 2. Clone your repo
For a private repo, create a GitHub Personal Access Token (Settings > Developer settings > Tokens, `repo` scope) and paste it below. Do NOT commit the token.

In [ ]:
GITHUB_USER = 'YOUR_USERNAME'
REPO = 'YOUR_REPO_NAME'
TOKEN = 'YOUR_TOKEN'   # keep this private

!git clone https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git /content/repo
%cd /content/repo
!pip install -q librosa soundfile

### 3. Download the dataset to Colab's local disk (~7 GB, takes a few minutes)

In [ ]:
%cd /content
!wget -q --show-progress https://os.unil.cloud.switch.ch/fma/fma_metadata.zip
!wget -q --show-progress https://os.unil.cloud.switch.ch/fma/fma_small.zip

In [ ]:
!unzip -q fma_metadata.zip
!unzip -q fma_small.zip
!ls /content

### 4. Load labels and verify

In [ ]:
import sys
sys.path.insert(0, '/content/repo')
from src.data import load_small_labels, audio_path, make_splits

METADATA_DIR = '/content/fma_metadata'
AUDIO_DIR = '/content/fma_small'

df = load_small_labels(METADATA_DIR)
print('Total tracks:', len(df))
print(df['genre'].value_counts())

Expected: 8000 tracks, 8 genres, 1000 each.

In [ ]:
# Check every audio file exists
missing = [t for t in df.index if not os.path.exists(audio_path(AUDIO_DIR, t))]
print('Missing audio files:', len(missing))

### 5. Create and save the 70/20/10 split

In [ ]:
df = make_splits(df)
print(df['split'].value_counts())
print(df.groupby(['split', 'genre']).size().unstack())

df.to_csv(f'{DRIVE_DIR}/splits.csv')
print('Saved to', f'{DRIVE_DIR}/splits.csv')